In [5]:
import pandas as pd


In [6]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
BASE_DIR = Path.cwd()
BASE_DIR
load_dotenv(BASE_DIR / ".env")
database_url = os.getenv(
    "DATABASE_URL",
    f"sqlite:///{BASE_DIR / 'data' / 'shoppulse.db'}"
)

database_url
engine = create_engine(database_url)

In [8]:
category_gap_df = pd.read_sql(query, engine)

category_gap_df


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct
0,Kitchen,3,0,50.0,0.0,50.0,0.00
1,Fitness,3,2,50.0,100.0,-50.0,66.67


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

query = """
WITH category_events AS (
    SELECT
        p.category,
        COUNT(CASE WHEN ce.event_type = 'product_view' THEN 1 END) AS product_views,
        COUNT(CASE WHEN ce.event_type = 'add_to_cart' THEN 1 END) AS add_to_cart_events
    FROM products p
    LEFT JOIN click_events ce
        ON p.id = ce.product_id
    GROUP BY p.category
),
totals AS (
    SELECT
        SUM(product_views) AS total_views,
        SUM(add_to_cart_events) AS total_add_to_cart
    FROM category_events
)
SELECT
    ce.category,
    ce.product_views,
    ce.add_to_cart_events,

    ROUND(100.0 * ce.product_views / NULLIF(t.total_views, 0), 2) AS view_share_pct,
    ROUND(100.0 * ce.add_to_cart_events / NULLIF(t.total_add_to_cart, 0), 2) AS add_to_cart_share_pct,

    ROUND(
        100.0 * ce.product_views / NULLIF(t.total_views, 0)
        - 100.0 * ce.add_to_cart_events / NULLIF(t.total_add_to_cart, 0),
        2
    ) AS view_vs_cart_share_gap_pct,

    ROUND(
        100.0 * ce.add_to_cart_events / NULLIF(ce.product_views, 0),
        2
    ) AS view_to_cart_rate_pct
FROM category_events ce
CROSS JOIN totals t
WHERE ce.product_views > 0
ORDER BY view_vs_cart_share_gap_pct DESC;
"""

category_gap_df = pd.read_sql(query, engine)


In [9]:
query_checkout_whatsapp = """
WITH category_carts AS (
    SELECT DISTINCT
        p.category,
        ce.cart_id
    FROM cart_events ce
    JOIN products p
        ON ce.product_id = p.id
    WHERE ce.event_type = 'add_to_cart'
      AND ce.cart_id IS NOT NULL
)

SELECT
    cc.category,
    COUNT(DISTINCT CASE WHEN ce.event_type = 'checkout_started' THEN ce.cart_id END) AS checkout_started_count,
    COUNT(DISTINCT CASE WHEN ce.event_type = 'whatsapp_order_click' THEN ce.cart_id END) AS whatsapp_order_count
FROM category_carts cc
LEFT JOIN cart_events ce
    ON cc.cart_id = ce.cart_id
GROUP BY cc.category;
"""

checkout_whatsapp_df = pd.read_sql(query_checkout_whatsapp, engine)

category_gap_df = category_gap_df.merge(
    checkout_whatsapp_df,
    on="category",
    how="left"
)

category_gap_df[["checkout_started_count", "whatsapp_order_count"]] = (
    category_gap_df[["checkout_started_count", "whatsapp_order_count"]]
    .fillna(0)
    .astype(int)
)

category_gap_df


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,checkout_started_count,whatsapp_order_count
0,Kitchen,3,0,50.0,0.0,50.0,0.00,0,0
1,Fitness,3,2,50.0,100.0,-50.0,66.67,1,1


In [10]:
category_gap_df["cart_to_whatsapp_click_ratio_pct"] = (
    100 * category_gap_df["whatsapp_order_count"]
    / category_gap_df["add_to_cart_events"].replace(0, pd.NA)
).round(2)

category_gap_df["cart_to_whatsapp_click_ratio_pct"] = (
    category_gap_df["cart_to_whatsapp_click_ratio_pct"].fillna(0)
)

category_gap_df


/var/folders/_c/1_53v6nj3sg2v1qskk_j_1d00000gn/T/ipykernel_2635/3746350167.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  category_gap_df["cart_to_whatsapp_click_ratio_pct"].fillna(0)


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,checkout_started_count,whatsapp_order_count,cart_to_whatsapp_click_ratio_pct
0,Kitchen,3,0,50.0,0.0,50.0,0.00,0,0,0.0
1,Fitness,3,2,50.0,100.0,-50.0,66.67,1,1,50.0


In [11]:
query_product_features = """
SELECT
    category,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS avg_price,
    ROUND(AVG(discount_percent), 2) AS avg_discount_pct,
    ROUND(AVG(rating), 2) AS avg_rating,
    SUM(review_count) AS total_reviews,
    ROUND(AVG(stock_quantity), 2) AS avg_stock_quantity,
    COUNT(CASE WHEN is_featured = 1 THEN 1 END) AS featured_product_count
FROM products
WHERE is_active = 1
GROUP BY category;
"""

product_features_df = pd.read_sql(query_product_features, engine)

category_gap_df = category_gap_df.merge(
    product_features_df,
    on="category",
    how="left"
)

category_gap_df


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,checkout_started_count,whatsapp_order_count,cart_to_whatsapp_click_ratio_pct,total_products,avg_price,avg_discount_pct,avg_rating,total_reviews,avg_stock_quantity,featured_product_count
0,Kitchen,3,0,50.0,0.0,50.0,0.00,0,0,0.0,5,979.0,19.0,4.36,1604,196.0,2
1,Fitness,3,2,50.0,100.0,-50.0,66.67,1,1,50.0,5,859.0,22.0,4.40,1015,154.0,2


In [12]:
query_session_features = """
SELECT
    p.category,
    COUNT(DISTINCT ce.session_id) AS unique_sessions,
    COUNT(DISTINCT ce.user_id) AS unique_users
FROM click_events ce
JOIN products p
    ON ce.product_id = p.id
WHERE ce.event_type IN ('product_view', 'add_to_cart')
GROUP BY p.category;
"""

session_features_df = pd.read_sql(query_session_features, engine)

category_gap_df = category_gap_df.merge(
    session_features_df,
    on="category",
    how="left"
)

category_gap_df[["unique_sessions", "unique_users"]] = (
    category_gap_df[["unique_sessions", "unique_users"]]
    .fillna(0)
    .astype(int)
)

category_gap_df


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,checkout_started_count,whatsapp_order_count,cart_to_whatsapp_click_ratio_pct,total_products,avg_price,avg_discount_pct,avg_rating,total_reviews,avg_stock_quantity,featured_product_count,unique_sessions,unique_users
0,Kitchen,3,0,50.0,0.0,50.0,0.00,0,0,0.0,5,979.0,19.0,4.36,1604,196.0,2,1,1
1,Fitness,3,2,50.0,100.0,-50.0,66.67,1,1,50.0,5,859.0,22.0,4.40,1015,154.0,2,1,1


In [13]:
category_gap_df["cart_to_checkout_ratio_pct"] = (
    100 * category_gap_df["checkout_started_count"]
    / category_gap_df["add_to_cart_events"].replace(0, pd.NA)
).round(2)

category_gap_df["checkout_to_whatsapp_ratio_pct"] = (
    100 * category_gap_df["whatsapp_order_count"]
    / category_gap_df["checkout_started_count"].replace(0, pd.NA)
).round(2)

category_gap_df[[
    "cart_to_checkout_ratio_pct",
    "checkout_to_whatsapp_ratio_pct"
]] = category_gap_df[[
    "cart_to_checkout_ratio_pct",
    "checkout_to_whatsapp_ratio_pct"
]].fillna(0)

category_gap_df


/var/folders/_c/1_53v6nj3sg2v1qskk_j_1d00000gn/T/ipykernel_2635/4240239693.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ]].fillna(0)


,category,product_views,add_to_cart_events,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,checkout_started_count,whatsapp_order_count,cart_to_whatsapp_click_ratio_pct,...,avg_price,avg_discount_pct,avg_rating,total_reviews,avg_stock_quantity,featured_product_count,unique_sessions,unique_users,cart_to_checkout_ratio_pct,checkout_to_whatsapp_ratio_pct
0,Kitchen,3,0,50.0,0.0,50.0,0.00,0,0,0.0,...,979.0,19.0,4.36,1604,196.0,2,1,1,0.0,0.0
1,Fitness,3,2,50.0,100.0,-50.0,66.67,1,1,50.0,...,859.0,22.0,4.40,1015,154.0,2,1,1,50.0,100.0


In [14]:
category_gap_df = category_gap_df[
    [
        "category",
        "product_views",
        "add_to_cart_events",
        "checkout_started_count",
        "whatsapp_order_count",
        "view_share_pct",
        "add_to_cart_share_pct",
        "view_vs_cart_share_gap_pct",
        "view_to_cart_rate_pct",
        "cart_to_checkout_ratio_pct",
        "checkout_to_whatsapp_ratio_pct",
        "cart_to_whatsapp_click_ratio_pct",
        "unique_sessions",
        "unique_users",
        "total_products",
        "avg_price",
        "avg_discount_pct",
        "avg_rating",
        "total_reviews",
        "avg_stock_quantity",
        "featured_product_count",
    ]
]

category_gap_df


,category,product_views,add_to_cart_events,checkout_started_count,whatsapp_order_count,view_share_pct,add_to_cart_share_pct,view_vs_cart_share_gap_pct,view_to_cart_rate_pct,cart_to_checkout_ratio_pct,...,cart_to_whatsapp_click_ratio_pct,unique_sessions,unique_users,total_products,avg_price,avg_discount_pct,avg_rating,total_reviews,avg_stock_quantity,featured_product_count
0,Kitchen,3,0,0,0,50.0,0.0,50.0,0.00,0.0,...,0.0,1,1,5,979.0,19.0,4.36,1604,196.0,2
1,Fitness,3,2,1,1,50.0,100.0,-50.0,66.67,50.0,...,50.0,1,1,5,859.0,22.0,4.40,1015,154.0,2
